# Advanced Certification in AIML
## A Program by IIIT-H and TalentSprint

Automated facial expression recognition provides an objective assessment of emotions. Human based assessment of emotions has many limitations and biases and automated facial expression technology has been found to deliver a better level of insight into behavior patterns. Emotion detection from facial expressions using AI is useful in automatically measuring consumers’ engagement with their content and brands, audience engagement for advertisements, customer satisfaction in the retail sector, psychological analyses, law enforcement etc.

**Objectives:**

**Stage 4 (20 Marks):** Train a CNN Model, update your HuggingFace Space repository, and test your Deployment for Expression Recognition on the HuggingFace Space App.

##**Stage 4 (20 Marks)**

**(i) Train a CNN Model for Expression Recognition on given Expression data**

**(ii) Deploy the Model and Perform Expression Recognition on Team Data through the HuggingFace Space App**


---


* Define and train a CNN for expression recognition for the data under folder `"Expression_data"` which segregated on expression basis.

* Collect your team data by running the code cells provided within the notebook below.

* Test your model on the collected team data and optimize the CNN architecture for predicting the respective labels of the images.

- Update files present in your cloned HuggingFace Space repository.

    - Save and Download the trained expression model (`expression_model.t7`) and upload/place it in your HuggingFace Space repository within **`app/Hackathon_setup/`** folder.
    
    - Update the model architecture in the **`app/Hackathon_setup/exp_recognition_model.py`** file.
    
    - Update the code in the **`get_expression()`** function of the **`app/Hackathon_setup/exp_recognition.py`** file. (See Deployment related files)

- Commit your changes and push to HuggingFace Space repository.

- Access the `App` tab of your repository to see the build progress (debug if error persists) [This step might take sometime.]

- Once the app has built successfully, you should see below message

    `Application startup complete. Uvicorn running on http://0.0.0.0:8001`

- Test the model's Expression Recognition functionality using the application running on your Space

    Go to 3-dots icon beside Settings, then select `Embed this Space` option, and go to `Direct URL`
    - Select the task, `Expression Recognition`
    - Select 'Send Anyway' when prompted
    - Upload your image and test
    - **NOTE:** When using the Direct URL link via android mobile, the camera option will also enable to capture images (Set 1:1 aspect ratio in camera settings before-hand)


### **Download the dataset**

In [1]:
#@title Run this cell to download the dataset

from IPython import get_ipython
ipython = get_ipython()

notebook="M3_Hackathon" #name of the notebook

def setup():
#  ipython.magic("sx pip3 install torch")
    ipython.magic("sx wget wget https://cdn.talentsprint.com/aiml/Experiment_related_data/Expression_data.zip")
    ipython.magic("sx unzip Expression_data.zip")
    ipython.magic("sx wget https://cdn.iisc.talentsprint.com/AIandMLOps/Datasets/lbpcascade_frontalface.xml")

    print ("Setup completed successfully")
    return

import os
if not os.path.exists(r'./Expression_data'):
  setup()

Setup completed successfully


In [2]:
%ls

Expression_data/     lbpcascade_frontalface.xml  sample_data/
Expression_data.zip  __MACOSX/


In [3]:
%ls Expression_data/

Facial_expression_test/  Facial_expression_train/


In [4]:
%ls Expression_data/Facial_expression_test

ANGER/  DISGUST/  FEAR/  HAPPINESS/  NEUTRAL/  SADNESS/  SURPRISE/


**Dataset attributes:**

During the setup you have downloaded the `Expression_data`:

* **Expression_data**: In this folder, the images are segregrated in terms of Expression
> * Expressions available: ANGER, DISGUST, FEAR, HAPPINESS, NEUTRAL, SADNESS, SURPRISE
> * Each class is organised as one folder
> * There are ~18000 total images in the training data and ~4500 total images in the testing data

### **Import Required Packages**

In [5]:
%matplotlib inline
import torchvision
import torchvision.datasets as dset
import torchvision.transforms as transforms
from torch.utils.data import DataLoader,Dataset
import matplotlib.pyplot as plt
import torchvision.utils
import numpy as np
import random
from PIL import Image                   # PIL (Pillow) is the Python Image Library. Used to cut and resize images, or do simple manipulation.
import torch
from torch.autograd import Variable
import PIL.ImageOps
import torch.nn as nn
from torch import optim
import torch.nn.functional as F
import os
import warnings
from time import sleep
import sys
warnings.filterwarnings('ignore')

For the following step, to obtain hints on building a CNN model for face expression, you may refer to this [article](https://drive.google.com/open?id=1P2rpaWW3tOtGGnw4dvtdZ4hjoc8iDNst).

### **Define Data Transformations and Custom Dataset**

In [6]:
IMG_SIZE = 48

# Define transformations for the training set
train_transforms = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

# Define transformations for the test set
test_transforms = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

# Define class to index mapping based on the folder names
# This assumes the folder names are the expression labels
all_labels = os.listdir('./Expression_data/Facial_expression_train')
expression_labels = sorted([label for label in all_labels if label != '.DS_Store'])
class_to_idx = {label: i for i, label in enumerate(expression_labels)}
idx_to_class = {i: label for i, label in enumerate(expression_labels)}

print("Class to Index Mapping:", class_to_idx)

Class to Index Mapping: {'ANGER': 0, 'DISGUST': 1, 'FEAR': 2, 'HAPPINESS': 3, 'NEUTRAL': 4, 'SADNESS': 5, 'SURPRISE': 6}


In [7]:
class ExpressionDataset(Dataset):
    def __init__(self, root_dir, transform=None, class_to_idx=None):
        self.root_dir = root_dir
        self.transform = transform
        self.class_to_idx = class_to_idx
        self.images = self._load_images_and_labels()

    def _load_images_and_labels(self):
        images = []
        for label_name in os.listdir(self.root_dir):
            label_path = os.path.join(self.root_dir, label_name)
            if os.path.isdir(label_path):
                label_idx = self.class_to_idx[label_name]
                for img_name in os.listdir(label_path):
                    img_path = os.path.join(label_path, img_name)
                    images.append((img_path, label_idx))
        return images

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):
        img_path, label = self.images[idx]
        image = Image.open(img_path).convert('RGB') # Convert to RGB to ensure 3 channels

        if self.transform:
            image = self.transform(image)

        return image, label

**Define and train a CNN model for expression recognition**

In [8]:
# Create Dataset instances
train_dataset = ExpressionDataset(root_dir='Expression_data/Facial_expression_train', transform=train_transforms, class_to_idx=class_to_idx)
test_dataset = ExpressionDataset(root_dir='Expression_data/Facial_expression_test', transform=test_transforms, class_to_idx=class_to_idx)

# Create DataLoader instances
train_dataloader = DataLoader(train_dataset, batch_size=64, shuffle=True, num_workers=2)
test_dataloader = DataLoader(test_dataset, batch_size=64, shuffle=False, num_workers=2)

print(f"Number of training samples: {len(train_dataset)}")
print(f"Number of testing samples: {len(test_dataset)}")

Number of training samples: 18178
Number of testing samples: 4548


In [9]:
#YOUR CODE HERE : Sample Helper function


In [10]:
#YOUR CODE HERE : Check number of training and Validation images
class ExpressionDetector(nn.Module):
    def __init__(self):
      super(ExpressionDetector, self).__init__()
      self._cnn = nn.Sequential(
            nn.ReflectionPad2d(1),       #Pads the input tensor using the reflection of the input boundary, it similar to the padding.
            nn.Conv2d(3, 6, kernel_size=3),
            nn.ReLU(inplace=True),
            nn.BatchNorm2d(6),
            nn.Dropout2d(0.2),
            nn.MaxPool2d(kernel_size=2, stride=2),

            nn.ReflectionPad2d(1),
            nn.Conv2d(6, 12, kernel_size=3),
            nn.ReLU(inplace=True),
            nn.BatchNorm2d(12),
            nn.Dropout2d(0.2),
            nn.MaxPool2d(kernel_size=2, stride=2),

            nn.ReflectionPad2d(1),
            nn.Conv2d(12, 24, kernel_size=3),
            nn.ReLU(inplace=True),
            nn.BatchNorm2d(24),
            nn.Dropout2d(0.2),
            nn.MaxPool2d(kernel_size=2, stride=2),

            nn.ReflectionPad2d(1),
            nn.Conv2d(24, 28, kernel_size=3),
            nn.ReLU(inplace=True),
            nn.BatchNorm2d(28), # Changed from 24 to 28 to match Conv2d output channels
            nn.Dropout2d(0.2),
            nn.MaxPool2d(kernel_size=2, stride=2)
      )
      self.fc = nn.Sequential(
            nn.Linear(28*3*3, 1000), # Adjusted input size to FC layer (48/2/2/2/2 = 3, 28 filters)
            nn.ReLU(inplace=True),

            nn.Linear(1000, 500),
            nn.ReLU(inplace=True),

            nn.Linear(500, 100),
            nn.ReLU(inplace=True),

            nn.Linear(100, 7)
      )

    def forward(self, image):
        output = self._cnn(image)
        output = output.view(output.size(0), -1) # Flatten the output for the fully connected layer
        output = self.fc(output)
        return output

In [11]:
#YOUR CODE HERE : Generate a batch of 10 images and labels


In [12]:
model = ExpressionDetector()

In [13]:
# Move model to GPU if available
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

ExpressionDetector(
  (_cnn): Sequential(
    (0): ReflectionPad2d((1, 1, 1, 1))
    (1): Conv2d(3, 6, kernel_size=(3, 3), stride=(1, 1))
    (2): ReLU(inplace=True)
    (3): BatchNorm2d(6, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (4): Dropout2d(p=0.2, inplace=False)
    (5): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (6): ReflectionPad2d((1, 1, 1, 1))
    (7): Conv2d(6, 12, kernel_size=(3, 3), stride=(1, 1))
    (8): ReLU(inplace=True)
    (9): BatchNorm2d(12, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (10): Dropout2d(p=0.2, inplace=False)
    (11): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (12): ReflectionPad2d((1, 1, 1, 1))
    (13): Conv2d(12, 24, kernel_size=(3, 3), stride=(1, 1))
    (14): ReLU(inplace=True)
    (15): BatchNorm2d(24, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (16): Dropout2d(p=0.2, inplace=False)
    (17): MaxPool2d

In [14]:
# YOUR CODE HERE : Print the summary of the model
import torchsummary
# Ensure torchsummary uses the correct device
torchsummary.summary(model, input_size=(3, 48, 48), device=str(device))

----------------------------------------------------------------
        Layer (type)               Output Shape         Param #
   ReflectionPad2d-1            [-1, 3, 50, 50]               0
            Conv2d-2            [-1, 6, 48, 48]             168
              ReLU-3            [-1, 6, 48, 48]               0
       BatchNorm2d-4            [-1, 6, 48, 48]              12
         Dropout2d-5            [-1, 6, 48, 48]               0
         MaxPool2d-6            [-1, 6, 24, 24]               0
   ReflectionPad2d-7            [-1, 6, 26, 26]               0
            Conv2d-8           [-1, 12, 24, 24]             660
              ReLU-9           [-1, 12, 24, 24]               0
      BatchNorm2d-10           [-1, 12, 24, 24]              24
        Dropout2d-11           [-1, 12, 24, 24]               0
        MaxPool2d-12           [-1, 12, 12, 12]               0
  ReflectionPad2d-13           [-1, 12, 14, 14]               0
           Conv2d-14           [-1, 24,

**Test your model and optimize CNN architecture for predicting the labels correctly**

In [15]:
# YOUR CODE HERE for test evaluation

In [16]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

In [17]:
def delete_saved_models():
  for file in os.listdir(r'./'):
    # Check if the file name contains 'expression_model' and ends with '.t7'
    # This is a safer check than just file.endswith('.t7') to avoid deleting unintended files.
    if 'expression_model' in file and file.endswith('.t7'):
      os.remove(file) # Corrected: use os.remove() directly
      print(f"Deleted {file}")

In [18]:
n_epochs = 50
train_losses = []
train_accuracy = []
best_validation_accuracy = 0

delete_saved_models()
for epoch in range(n_epochs):
    train_loss = 0
    correct_pred = 0
    for images, labels in train_dataloader:
        #Move to CUDA
        images = images.to(device)
        labels = labels.to(device)

        #Zero out gradients
        optimizer.zero_grad()

        #forward pass
        output = model(images)

        #loss computation
        loss = criterion(output, labels)
        train_loss += loss.item() # Corrected: accumulate loss

        #Back propogation
        loss.backward()

        #Optimizer step
        optimizer.step()

        _,predicted = torch.max(output.data, 1)
        correct_pred += (predicted == labels).sum().item()

    train_accuracy.append(100 * correct_pred / len(train_dataset))
    train_losses.append(train_loss/len(train_dataloader)) # Divide by len(train_dataloader) for average batch loss
    print(f"Epochs :{epoch+1}/{n_epochs} train_loss : {train_loss/len(train_dataloader):0.4f} train_accuracy : {100 * correct_pred / len(train_dataset):0.3f}")

    model.eval()
    val_loss = 0
    val_acc = 0
    with torch.no_grad():
        for images, labels in test_dataloader:
            #Move to CUDA
            images = images.to(device)
            labels = labels.to(device)
            y_predicted = model(images)
            loss_val = criterion(y_predicted, labels)
            val_loss += loss_val.item()

            _,predicted = torch.max(model(images).data, 1)
            val_acc += (predicted == labels).sum().item()

    val_loss /=len(test_dataloader.dataset)
    val_acc /=len(test_dataloader.dataset)
    print(f"validation_loss : {val_loss/len(train_dataloader):0.4f} validation_accuracy : {100 * val_acc:0.3f}")
    if val_acc > best_validation_accuracy:
      print(f"Best validation Accuracy: {val_acc:0.4f} at epoch: {epoch}. Saving Model")
      best_validation_accuracy = val_acc
      filename = f'expression_model_{epoch}_{val_acc*100:0.2f}.t7'
      torch.save(model.state_dict(), filename )
      from google.colab import files
      files.download(filename)

Epochs :1/50 train_loss : 1.7598 train_accuracy : 30.119
validation_loss : 0.0001 validation_accuracy : 32.366
Best validation Accuracy: 0.3237 at epoch: 0. Saving Model


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Epochs :2/50 train_loss : 1.7104 train_accuracy : 33.040
validation_loss : 0.0001 validation_accuracy : 35.554
Best validation Accuracy: 0.3555 at epoch: 1. Saving Model


<IPython.core.display.Javascript object>

...

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Epochs :21/50 train_loss : 1.1729 train_accuracy : 55.804
validation_loss : 0.0001 validation_accuracy : 41.557
Epochs :22/50 train_loss : 1.1588 train_accuracy : 57.091
validation_loss : 0.0001 validation_accuracy : 41.645
Epochs :23/50 train_loss : 1.1142 train_accuracy : 58.527
validation_loss : 0.0001 validation_accuracy : 41.095
Epochs :24/50 train_loss : 1.0826 train_accuracy : 59.930
validation_loss : 0.0001 validation_accuracy : 41.271
Epochs :25/50 train_loss : 1.0444 train_accuracy : 60.986
validation_loss : 0.0001 validation_accuracy : 41.799
Epochs :26/50 train_loss : 1.0121 train_accuracy : 62.625
validation_loss : 0.0001 validation_accuracy : 41.315
Epochs :27/50 train_loss : 0.9698 train_accuracy : 64.242
validation_loss : 0.0001 validation_accuracy : 40.150
Epochs :28/50 train_loss : 0.9255 train_accuracy : 65.827
validation_loss : 0.0001 validation_accuracy : 39.732
Epochs :29/50 train_loss : 0.8901 train_accuracy : 67.048
validation_loss : 0.0001 validation_accuracy :

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [19]:
from torchvision import transforms, models

inception = models.inception_v3(pretrained=True)

Downloading: "https://download.pytorch.org/models/inception_v3_google-0cc3c7bd.pth" to /root/.cache/torch/hub/checkpoints/inception_v3_google-0cc3c7bd.pth


100%|██████████| 104M/104M [00:00<00:00, 148MB/s]


In [30]:
import torchsummary
torchsummary.summary(inception, input_size=(3, 299, 299), device=str(device))

----------------------------------------------------------------
        Layer (type)               Output Shape         Param #
            Conv2d-1         [-1, 32, 149, 149]             864
       BatchNorm2d-2         [-1, 32, 149, 149]              64
       BasicConv2d-3         [-1, 32, 149, 149]               0
            Conv2d-4         [-1, 32, 147, 147]           9,216
       BatchNorm2d-5         [-1, 32, 147, 147]              64
       BasicConv2d-6         [-1, 32, 147, 147]               0
            Conv2d-7         [-1, 64, 147, 147]          18,432
       BatchNorm2d-8         [-1, 64, 147, 147]             128
       BasicConv2d-9         [-1, 64, 147, 147]               0
        MaxPool2d-10           [-1, 64, 73, 73]               0
           Conv2d-11           [-1, 80, 73, 73]           5,120
      BatchNorm2d-12           [-1, 80, 73, 73]             160
      BasicConv2d-13           [-1, 80, 73, 73]               0
           Conv2d-14          [-1, 192,

In [31]:
classifier = list(inception.modules())[-1]
new_classifier_layer = []
new_classifier_layer.append(nn.Linear(classifier.in_features, 2000))
new_classifier_layer.append(nn.Linear(2000, 1000))
new_classifier_layer.append(nn.Linear(1000, 7))
inception.fc = nn.Sequential(*new_classifier_layer)

In [ ]:
_epochs = 50
train_losses = []
train_accuracy = []
best_validation_accuracy = 0

delete_saved_models()
for epoch in range(n_epochs):
    train_loss = 0
    correct_pred = 0
    for images, labels in train_dataloader:
        #Move to CUDA
        images = images.to(device)
        labels = labels.to(device)

        #Zero out gradients
        optimizer.zero_grad()

        #forward pass
        output = inception(images)

        #loss computation
        loss = criterion(output, labels)
        train_loss += loss.item() # Corrected: accumulate loss

        #Back propogation
        loss.backward()

        #Optimizer step
        optimizer.step()

        _,predicted = torch.max(output.data, 1)
        correct_pred += (predicted == labels).sum().item()

    train_accuracy.append(100 * correct_pred / len(train_dataset))
    train_losses.append(train_loss/len(train_dataloader)) # Divide by len(train_dataloader) for average batch loss
    print(f"Epochs :{epoch+1}/{n_epochs} train_loss : {train_loss/len(train_dataloader):0.4f} train_accuracy : {100 * correct_pred / len(train_dataset):0.3f}")

    inception.eval()
    val_loss = 0
    val_acc = 0
    with torch.no_grad():
        for images, labels in test_dataloader:
            #Move to CUDA
            images = images.to(device)
            labels = labels.to(device)
            y_predicted = inception(images)
            loss_val = criterion(y_predicted, labels)
            val_loss += loss_val.item()

            _,predicted = torch.max(inception(images).data, 1)
            val_acc += (predicted == labels).sum().item()

    val_loss /=len(test_dataloader.dataset)
    val_acc /=len(test_dataloader.dataset)
    print(f"validation_loss : {val_loss/len(train_dataloader):0.4f} validation_accuracy : {100 * val_acc:0.3f}")
    if val_acc > best_validation_accuracy:
      print(f"Best validation Accuracy: {val_acc:0.4f} at epoch: {epoch}. Saving Model")
      best_validation_accuracy = val_acc
      filename = f'expression_model_{epoch}_{val_acc*100:0.2f}.t7'
      torch.save(model.state_dict(), filename )
      from google.colab import files
      files.download(filename)

#### **Team Data Collection**

**Collect your team data and fine-tune the CNN for expression data on your team**

- Collect Team Data by running the code cells provided below
    - The collected Expression images of your team will be stored in the `captured_images_with_Expression` directory

    - NOTE: *Since team members will be using separate colab notebooks, they can capture their own face images and then download it and share with other members for model testing.* Code cell is provided below to download the data.

- This data will be useful for testing the above trained cnn network

In [ ]:
# @title Run this cell to Setup Image Capturing in Colab {display-mode: "form"}

from IPython.display import display, Javascript
from google.colab.output import eval_js
from base64 import b64decode
from PIL import Image
import imageio
import datetime
import pathlib
import cv2
import numpy as np
import matplotlib.pyplot as plt

AREA_THRESHOLD = 2304

def take_photo(filename='photo.jpg', quality=0.8):
  js = Javascript('''
    async function takePhoto(quality) {
      const div = document.createElement('div');
      const capture = document.createElement('button');
      capture.textContent = 'Capture';
      div.appendChild(capture);

      const video = document.createElement('video');
      video.style.display = 'block';
      const stream = await navigator.mediaDevices.getUserMedia({video: true});

      document.body.appendChild(div);
      div.appendChild(video);
      video.srcObject = stream;
      await video.play();

      // Resize the output to fit the video element.
      google.colab.output.setIframeHeight(document.documentElement.scrollHeight, true);

      // Wait for Capture to be clicked.
      await new Promise((resolve) => capture.onclick = resolve);

      const canvas = document.createElement('canvas');
      canvas.width = video.videoWidth;
      canvas.height = video.videoHeight;
      canvas.getContext('2d').drawImage(video, 0, 0);
      stream.getVideoTracks()[0].stop();
      div.remove();
      return canvas.toDataURL('image/jpeg', quality);
    }
    ''')
  display(js)
  data = eval_js('takePhoto({})'.format(quality))
  binary = b64decode(data.split(',')[1])
  with open(filename, 'wb') as f:
    f.write(binary)

  im = Image.open(filename)
  im1 = im.crop((80, 0, 560, 480))
  im1.save(filename)

  return filename


def save_faces(miniframe, filepath):
    TRAINSET = "lbpcascade_frontalface.xml"
    classifier = cv2.CascadeClassifier(TRAINSET)
    faces = classifier.detectMultiScale(miniframe)
    image = get_large_face(miniframe, faces)
    if not isinstance(image, np.ndarray):
        return {"status" : False}
    plt.imshow(image)
    plt.show()
    #cv2.imwrite(filepath, image)
    imageio.imwrite(filepath, image)
    return {"status" : True}

def get_large_face(miniframe, faces):
    images = []
    face_areas = []
    required_image = 0
    for x,y,w,h in faces:
        face_cropped = miniframe[y:y+h, x:x+w]
        face_areas.append(w*h)
        images.append(face_cropped)
        required_image = images[np.argmax(face_areas)]
    if not face_areas:
        return 0
    if face_areas[np.argmax(face_areas)] < AREA_THRESHOLD:
        return 0

    return required_image

def save_image(filename, class_name):
    base_path = "captured_images_with_Expression/"

    pathlib.Path(base_path + class_name).mkdir(parents=True, exist_ok=True)
    file_name = class_name + "_" + datetime.datetime.now().strftime("%s") + ".jpg"
    filepath = base_path +class_name + "/" + file_name

    image = Image.open(filename)
    miniframe = np.asarray(image)
    status = save_faces(miniframe, filepath)
    if status['status']:
        print("Image saved in " + filepath, flush = True)
    else:
        print("Face not found!\nRetry!")


In [ ]:
# @title Capture an Image for ANGER $ $  [Re-run this cell to capture another image] {display-mode: "form"}

class_name = "ANGER"

from IPython.display import Image as IPyImage
try:
  filename = take_photo()
  #print('Saved to {}'.format(filename))
  # Show the image which was just taken.
  #display(IPyImage(filename))
except Exception as err:
  # Errors will be thrown if the user does not have a webcam or if they do not
  # grant the page permission to access it.
  print(str(err))

save_image(filename, class_name)

In [ ]:
# @title Capture an Image for DISGUST $ $  [Re-run this cell to capture another image] {display-mode: "form"}

class_name = "DISGUST"

from IPython.display import Image as IPyImage
try:
  filename = take_photo()
  #print('Saved to {}'.format(filename))
  # Show the image which was just taken.
  #display(IPyImage(filename))
except Exception as err:
  # Errors will be thrown if the user does not have a webcam or if they do not
  # grant the page permission to access it.
  print(str(err))

save_image(filename, class_name)

In [ ]:
# @title Capture an Image for FEAR $ $  [Re-run this cell to capture another image] {display-mode: "form"}

class_name = "FEAR"

from IPython.display import Image as IPyImage
try:
  filename = take_photo()
  #print('Saved to {}'.format(filename))
  # Show the image which was just taken.
  #display(IPyImage(filename))
except Exception as err:
  # Errors will be thrown if the user does not have a webcam or if they do not
  # grant the page permission to access it.
  print(str(err))

save_image(filename, class_name)

In [ ]:
# @title Capture an Image for HAPPINESS $ $  [Re-run this cell to capture another image] {display-mode: "form"}

class_name = "HAPPINESS"

from IPython.display import Image as IPyImage
try:
  filename = take_photo()
  #print('Saved to {}'.format(filename))
  # Show the image which was just taken.
  #display(IPyImage(filename))
except Exception as err:
  # Errors will be thrown if the user does not have a webcam or if they do not
  # grant the page permission to access it.
  print(str(err))

save_image(filename, class_name)

In [ ]:
# @title Capture an Image for NEUTRAL $ $  [Re-run this cell to capture another image] {display-mode: "form"}

class_name = "NEUTRAL"

from IPython.display import Image as IPyImage
try:
  filename = take_photo()
  #print('Saved to {}'.format(filename))
  # Show the image which was just taken.
  #display(IPyImage(filename))
except Exception as err:
  # Errors will be thrown if the user does not have a webcam or if they do not
  # grant the page permission to access it.
  print(str(err))

save_image(filename, class_name)

In [ ]:
# @title Capture an Image for SADNESS $ $  [Re-run this cell to capture another image] {display-mode: "form"}

class_name = "SADNESS"

from IPython.display import Image as IPyImage
try:
  filename = take_photo()
  #print('Saved to {}'.format(filename))
  # Show the image which was just taken.
  #display(IPyImage(filename))
except Exception as err:
  # Errors will be thrown if the user does not have a webcam or if they do not
  # grant the page permission to access it.
  print(str(err))

save_image(filename, class_name)

In [ ]:
# @title Capture an Image for SURPRISE $ $  [Re-run this cell to capture another image] {display-mode: "form"}

class_name = "SURPRISE"

from IPython.display import Image as IPyImage
try:
  filename = take_photo()
  #print('Saved to {}'.format(filename))
  # Show the image which was just taken.
  #display(IPyImage(filename))
except Exception as err:
  # Errors will be thrown if the user does not have a webcam or if they do not
  # grant the page permission to access it.
  print(str(err))

save_image(filename, class_name)

In [ ]:
# @title Download Collected Images $ $ [OPTIONAL]  {display-mode: "form"}

from google.colab import files
!zip -r "captured_images_with_Expression.zip" "captured_images_with_Expression"
files.download('captured_images_with_Expression.zip')
print("Downloaded captured_images_with_Expression.zip !!")

In [ ]:
%ls

**While uploading the team images manually to captured_images. this file .ipynb_checkpoints will be created and make issue to delet run the below code**

In [ ]:
rm -rf `find -type d -name .ipynb_checkpoints`

In [ ]:
# YOUR CODE HERE for loading the team expression data. Note: Use the same transform which used for Expression_Data.
# YOU CODE HERE for Dataloader

In [ ]:
# YOUR CODE HERE for getting the CNN representation of your team data with expression. Optimize the CNN model for predicting the labels of expressions correctly
# Note: If the CNN Model is not performing as expected, then you can add your Team Data to the Existing Training Data and Re-Train the Model.

**Save your trained model**

* Save the state dictionary of the classifier (use pytorch only), It will be useful in
integrating model to the mobile app

 [Hint](https://pytorch.org/tutorials/beginner/saving_loading_models.html)

In [ ]:
### YOUR CODE HERE for saving the CNN model

**Download your trained model**
* Given the path of model file the following code downloads it through the browser

In [ ]:
from google.colab import files
files.download('expression_model.t7')
#files.download('<model_file_path>')